Partie1: Evaluation de Rag

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# ================================
# 🔹 1. CHEMINS DES PDF
# ================================

pdf_files = ["../file1.pdf", "../file2.pdf", "../file3.pdf"]



# ================================
# 🔹 2. CHARGEMENT DES DOCUMENTS
# ================================

all_docs = []

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    all_docs.extend(docs)

print(f"Nombre total de pages : {len(all_docs)}")


# ================================
# 🔹 3. CHUNKING (PARTIE CRITIQUE DU RAG)
# ================================

# 🎯 Pourquoi chunk_size = 400 ?
# → petits chunks = meilleure précision de recherche
# → mais trop petits = perte de contexte
# → 400 est un compromis (bon pour documents moyens)

# 🎯 Pourquoi overlap = 50 ?
# → permet de garder une continuité entre chunks
# → évite de couper une phrase ou une idée importante
# → 50 = léger overlap (évite redondance excessive)

# ⚠️ Impact réel :
# - chunk_size trop grand → réponses vagues
# - chunk_size trop petit → réponses incomplètes
# - overlap trop grand → duplication inutile
# - overlap trop faible → perte d'information

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]  # découpe intelligente (paragraphes → mots)
)

chunks = text_splitter.split_documents(all_docs)

print(f"Nombre total de chunks : {len(chunks)}")


# ================================
# 🔹 4. EMBEDDINGS
# ================================

# 🎯 Modèle choisi : thenlper/gte-small
# → rapide
# → léger
# → vecteurs de dimension 384
# → bon compromis performance / vitesse

embedding_model = HuggingFaceEmbeddings(
    model_name="thenlper/gte-small",
    model_kwargs={"device": "cpu"}  
)


# ================================
# 🔹 5. BASE VECTORIELLE (CHROMA)
# ================================

persist_dir = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_dir,
    collection_name="rag_collection"
)


# ================================
# 🔹 6. RETRIEVER
# ================================

# 🎯 k = 3 → nombre de chunks récupérés
# → petit k = précis mais limité
# → grand k = plus de contexte mais bruit

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Indexation terminée.")

Nombre total de pages : 46
Nombre total de chunks : 215


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexation terminée.


Étape 2 : Charger le dataset de questions

In [ ]:
import json

with open("questionSource.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Extraire toutes les paires question-réponse dans une liste simple
dataset = []
for doc in data:
    for qa in doc["qa_pairs"]:
        dataset.append({
            "question": qa["question"],
            "answer": qa["answer"]
        })

print(f"Nombre de questions : {len(dataset)}") 

Nombre de questions : 30


Étape 3 :  Génération de la baseline (modèle principal)


Ollama

In [22]:
import json
import ollama
import time

with open("../questionSource.json", "r", encoding="utf-8") as f:
    source_data = json.load(f)

results_data = []

for doc in source_data:
    doc_copy = {"document": doc["document"], "qa_pairs": []}
    for qa in doc["qa_pairs"]:
        question = qa["question"]
        # Recherche vectorielle (utilise votre retriever existant)
        docs = retriever.invoke(question)
        context = "\n\n".join([d.page_content for d in docs])
        prompt = f"""Utilise le contexte ci-dessous pour répondre à la question.
Si tu ne connais pas la réponse, dis simplement "Je ne sais pas".

Contexte :
{context}

Question : {question}
Réponse :"""
        response = ollama.chat(model="gemma3", messages=[{"role": "user", "content": prompt}])
        answer = response["message"]["content"]
        doc_copy["qa_pairs"].append({"question": question, "answer": answer})
        time.sleep(0.2)   # petite pause pour éviter de surcharger
    results_data.append(doc_copy)

with open("../questionResultsGemm.json", "w", encoding="utf-8") as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

print("✅ questionResultsGemm.json sauvegardé avec succès (via Ollama).")

✅ questionResultsGemm.json sauvegardé avec succès (via Ollama).


OpenRouter

In [ ]:
import json
import time
from openai import OpenAI

OPENROUTER_API_KEY = "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"

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# Charger le fichier source (structure avec documents et qa_pairs)
with open("../questionSource.json", "r", encoding="utf-8") as f:
    source_data = json.load(f)

results_data = []

# Parcourir chaque document
for doc in source_data:
    doc_name = doc["document"]
    doc_copy = {"document": doc_name, "qa_pairs": []}
    
    for qa in doc["qa_pairs"]:
        question = qa["question"]
        
        # Recherche vectorielle (retriever déjà défini)
        docs = retriever.invoke(question)
        context = "\n\n".join([d.page_content for d in docs])
        
        if not context.strip():
            answer = "Je ne sais pas"
        else:
            prompt = f"""
Utilise le contexte pour répondre.
Si inconnu → "Je ne sais pas".

Contexte :
{context}

Question : {question}
Réponse :
"""
            try:
                response = client.chat.completions.create(
                    model="arcee-ai/trinity-large-preview:free",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0
                )
                answer = response.choices[0].message.content
            except Exception as e:
                answer = f"Erreur : {e}"
        
        doc_copy["qa_pairs"].append({
            "question": question,
            "answer": answer
        })
        time.sleep(0.5)   # pour respecter les limites de taux
    
    results_data.append(doc_copy)

# Sauvegarder le fichier
with open("../questionResults.json", "w", encoding="utf-8") as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

print("✅ questionResults.json sauvegardé avec succès.")

✅ questionResults.json sauvegardé avec succès.


Étape 4 : Évaluation des réponses


In [20]:
import json
import re
import ollama

def evaluate_answers(source_file, results_file, judge_model="gemma3"):
    """
    Évalue les réponses générées par rapport aux réponses de référence.
    Les deux fichiers ont la même structure : liste de documents contenant "qa_pairs".
    """
    with open(source_file, "r", encoding="utf-8") as f:
        source = json.load(f)
    with open(results_file, "r", encoding="utf-8") as f:
        results = json.load(f)

    # Extraire toutes les paires dans l'ordre (document par document, paire par paire)
    ref_answers = []
    gen_answers = []
    for doc_s, doc_r in zip(source, results):
        for qa_s, qa_r in zip(doc_s["qa_pairs"], doc_r["qa_pairs"]):
            ref_answers.append(qa_s["answer"])
            gen_answers.append(qa_r["answer"])

    scores = []
    for ref, gen in zip(ref_answers, gen_answers):
        eval_prompt = f"""Évalue la réponse générée par rapport à la réponse de référence.
Attribue une note de 1 à 5 selon les critères :
5 : très fidèle, identique ou paraphrase exacte.
4 : correct, contient les informations principales.
3 : partiel, manque certains détails importants.
2 : faible, informations approximatives ou hors sujet.
1 : incorrect, complètement à côté.

Réponse de référence : {ref}
Réponse générée : {gen}

Note (1-5) :"""
        eval_response = ollama.chat(
            model=judge_model,
            messages=[{"role": "user", "content": eval_prompt}]
        )
        match = re.search(r'\b[1-5]\b', eval_response["message"]["content"])
        score = int(match.group()) if match else 3
        scores.append(score)

    global_score = sum(scores) / len(scores)
    return scores, global_score

# Évaluation des réponses générées
scores, global_score = evaluate_answers("questionSource.json", "questionResultsOll.json")
print("Scores individuels :", scores)
print("Score global :", global_score)

# Sauvegarde des scores
with open("scoresOll.json", "w", encoding="utf-8") as f:
    json.dump({"individual_scores": scores, "global_score": global_score}, f, indent=2)

Scores individuels : [4, 4, 4, 4, 4, 2, 3, 4, 1, 2, 4, 1, 4, 4, 2, 3, 4, 1, 4, 4, 5, 2, 4, 4, 4, 5, 5, 3, 2, 3]
Score global : 3.3333333333333335


OpenRouter

In [ ]:
import json
import re
import time
from openai import OpenAI

# Configuration OpenRouter 
client = OpenAI(
    api_key="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",
    base_url="https://openrouter.ai/api/v1"
)

def evaluate_answers_nvidia(source_file, results_file, judge_model="nvidia/llama-3.1-nemotron-70b-instruct"):
    """
    Évalue les réponses générées en utilisant un modèle NVIDIA via OpenRouter.
    Le modèle par défaut est nvidia/llama-3.1-nemotron-70b-instruct (payant, crédits nécessaires).
    Vous pouvez le remplacer par un modèle gratuit comme "nvidia/llama-3.1-nemotron-nano-8b-v1:free"
    si disponible.
    """
    with open(source_file, "r", encoding="utf-8") as f:
        source = json.load(f)
    with open(results_file, "r", encoding="utf-8") as f:
        results = json.load(f)

    ref_answers = []
    gen_answers = []
    for doc_s, doc_r in zip(source, results):
        for qa_s, qa_r in zip(doc_s["qa_pairs"], doc_r["qa_pairs"]):
            ref_answers.append(qa_s["answer"])
            gen_answers.append(qa_r["answer"])

    scores = []
    for ref, gen in zip(ref_answers, gen_answers):
        eval_prompt = f"""Évalue la réponse générée par rapport à la réponse de référence.
Attribue une note de 1 à 5 selon les critères :
5 : très fidèle, identique ou paraphrase exacte.
4 : correct, contient les informations principales.
3 : partiel, manque certains détails importants.
2 : faible, informations approximatives ou hors sujet.
1 : incorrect, complètement à côté.

Réponse de référence : {ref}
Réponse générée : {gen}

Note (1-5) :"""
        try:
            response = client.chat.completions.create(
                model=judge_model,
                messages=[{"role": "user", "content": eval_prompt}],
                temperature=0
            )
            content = response.choices[0].message.content
            match = re.search(r'\b[1-5]\b', content)
            score = int(match.group()) if match else 3
        except Exception as e:
            print(f"Erreur d'évaluation : {e}")
            score = 3
        scores.append(score)
        time.sleep(1.0)   # Pause pour éviter les limites de taux

    global_score = sum(scores) / len(scores)
    return scores, global_score

# Choix du modèle (remplacez par un modèle valide)
judge_model = "nvidia/llama-3.1-nemotron-70b-instruct"   
# judge_model = "nvidia/llama-3.1-nemotron-nano-8b-v1"   

scores, global_score = evaluate_answers_nvidia("questionSource.json", "questionResults.json", judge_model=judge_model)
print("Scores individuels :", scores)
print(f"Score global : {global_score:.2f}/5")

# Sauvegarde
with open("scores_nvidia.json", "w", encoding="utf-8") as f:
    json.dump({"individual_scores": scores, "global_score": global_score}, f, indent=2)

Scores individuels : [5, 5, 5, 5, 5, 4, 4, 4, 4, 3, 5, 4, 4, 5, 5, 5, 5, 5, 3, 5, 5, 5, 4, 5, 5, 5, 5, 3, 2, 4]
Score global : 4.43/5


Evaluation avec des metriques: 

In [30]:
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_answers_embedding(source_file, results_file, embedding_model):
    """
    Évalue les réponses en comparant les embeddings des réponses générées et de référence.
    Retourne un score de 1 à 5 basé sur la similarité cosinus.
    """
    with open(source_file, "r", encoding="utf-8") as f:
        source = json.load(f)
    with open(results_file, "r", encoding="utf-8") as f:
        results = json.load(f)

    ref_answers = []
    gen_answers = []
    for doc_s, doc_r in zip(source, results):
        for qa_s, qa_r in zip(doc_s["qa_pairs"], doc_r["qa_pairs"]):
            ref_answers.append(qa_s["answer"])
            gen_answers.append(qa_r["answer"])

    scores = []
    for ref, gen in zip(ref_answers, gen_answers):
        emb_ref = embedding_model.embed_query(ref)
        emb_gen = embedding_model.embed_query(gen)
        # Calcul de la similarité cosinus
        sim = cosine_similarity([emb_ref], [emb_gen])[0][0]
        # Conversion similarité -> score 1-5
        if sim >= 0.9:
            score = 5
        elif sim >= 0.8:
            score = 4
        elif sim >= 0.7:
            score = 3
        elif sim >= 0.6:
            score = 2
        else:
            score = 1
        scores.append(score)

    global_score = sum(scores) / len(scores)
    return scores, global_score

# ================================
# 🔹 Évaluation (avec embeddings)
# ================================
# Utilisez le même embedding_model que pour l'indexation
scores, global_score = evaluate_answers_embedding("../questionSource.json", "../questionResults.json", embedding_model)
print("Scores individuels :", scores)
print("Score global :", global_score)

# Sauvegarde
with open("../scores_embedding.json", "w", encoding="utf-8") as f:
    json.dump({"individual_scores": scores, "global_score": global_score}, f, indent=2)

Scores individuels : [5, 4, 5, 5, 5, 4, 4, 4, 5, 4, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 4, 5]
Score global : 4.733333333333333


Étape 5 : Optimisation

Open Router:

Test

In [ ]:
import json
import re
import ollama
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma

# ------------------------------------------------------------------------------
# Test des différents modèles d'embedding
# ------------------------------------------------------------------------------

# 1. Charger les documents si nécessaire
if 'all_docs' not in locals():
    print("Chargement des PDF...")
    pdf_files = ["../file1.pdf", "../file2.pdf", "../file3.pdf"]
    all_docs = []
    for pdf in pdf_files:
        try:
            loader = PyPDFLoader(pdf)
            docs = loader.load()
            all_docs.extend(docs)
            print(f"  - {pdf} chargé ({len(docs)} pages)")
        except Exception as e:
            print(f"  - Erreur pour {pdf} : {e}")
    print(f"Nombre total de pages : {len(all_docs)}")

# 2. Définition des modèles à tester
EMBEDDING_MODELS = {
    "gte-small (léger)":        "thenlper/gte-small",           # 384 dims
    "gte-large (précis)":       "thenlper/gte-large",           # 1024 dims
    "MiniLM (rapide)":          "sentence-transformers/all-MiniLM-L6-v2",  # 384 dims
    "multilingual (fr/en)":     "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "intfloat/e5-base-v2":      "intfloat/e5-base-v2",         # SOTA retrieval
}

# 3. Paramètres fixes pour le test
test_cs = 400
test_ov = 16
test_k = 5

# 4. Charger le dataset si nécessaire
if 'dataset_flat' not in locals():
    with open("../questionSource.json", "r", encoding="utf-8") as f:
        source_data = json.load(f)
    dataset_flat = []
    for doc in source_data:
        for qa in doc["qa_pairs"]:
            dataset_flat.append({"question": qa["question"], "answer": qa["answer"]})

subset = dataset_flat[:5]  # 5 questions pour le test rapide

print(f"Configuration de test : chunk_size={test_cs}, overlap={test_ov}, k={test_k}")
print(f"Nombre de questions pour le test : {len(subset)}")
print(f"Nombre total de pages : {len(all_docs)}")

# 5. Fonction de test pour un embedding model
def test_embedding_model(embedding_model_name, embedding_model_id, dataset_subset, all_docs, llm_model="gemma3", judge_model="gemma3"):
    """Teste un modèle d'embedding sur le sous-ensemble de questions."""
    print(f"  - Chargement du modèle : {embedding_model_name}")
    
    try:
        # Charger le modèle d'embedding
        emb_model = HuggingFaceEmbeddings(
            model_name=embedding_model_id,
            model_kwargs={"device": "cpu"}
        )
        
        # Découpage des documents
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=test_cs,
            chunk_overlap=test_ov,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(all_docs)
        
        # Création de l'index Chroma
        persist_dir = f"chroma_test_emb_{embedding_model_id.replace('/', '_')[-30:]}"
        vs = Chroma.from_documents(chunks, emb_model, persist_directory=persist_dir, collection_name="test")
        ret = vs.as_retriever(search_kwargs={"k": test_k})
        
        # Génération des réponses
        results = []
        for item in dataset_subset:
            question = item["question"]
            docs = ret.invoke(question)
            context = "\n\n".join([d.page_content for d in docs])
            prompt = f"""Utilise le contexte ci-dessous pour répondre à la question.
Si tu ne connais pas la réponse, dis simplement "Je ne sais pas".

Contexte :
{context}

Question : {question}
Réponse :"""
            response = ollama.chat(model=llm_model, messages=[{"role": "user", "content": prompt}])
            results.append({"question": question, "answer": response["message"]["content"]})
        
        # Évaluation
        scores = []
        for i, item in enumerate(dataset_subset):
            ref_answer = item["answer"]
            gen_answer = results[i]["answer"]
            eval_prompt = f"""Note de 1 à 5 la réponse générée par rapport à la référence.
Référence : {ref_answer}
Générée : {gen_answer}
Note :"""
            eval_resp = ollama.chat(model=judge_model, messages=[{"role": "user", "content": eval_prompt}])
            match = re.search(r'\b[1-5]\b', eval_resp["message"]["content"])
            score = int(match.group()) if match else 3
            scores.append(score)
        
        # Nettoyage
        vs.delete_collection()
        
        avg_score = sum(scores) / len(scores)
        return avg_score, scores
        
    except Exception as e:
        print(f"  - Erreur lors du test : {e}")
        raise e

# 6. Boucle de test sur tous les modèles d'embedding
print("\n" + "="*60)
print("TEST DES MODÈLES D'EMBEDDING")
print("="*60)

embedding_results = {}

for name, model_id in EMBEDDING_MODELS.items():
    print(f"\n📊 Test du modèle : {name}")
    print(f"   ID : {model_id}")
    
    try:
        start_time = time.time()
        avg_score, scores = test_embedding_model(
            embedding_model_name=name,
            embedding_model_id=model_id,
            dataset_subset=subset,
            all_docs=all_docs,
            llm_model="gemma3",
            judge_model="gemma3"
        )
        elapsed = time.time() - start_time
        
        embedding_results[name] = {
            "model_id": model_id,
            "avg_score": avg_score,
            "scores": scores,
            "time": elapsed
        }
        
        print(f"   ✅ Score moyen : {avg_score:.2f}/5")
        print(f"   ⏱️  Temps : {elapsed:.1f} secondes")
        print(f"   📊 Scores individuels : {scores}")
        
    except Exception as e:
        print(f"   ❌ Erreur : {e}")
        embedding_results[name] = {
            "model_id": model_id,
            "avg_score": 0,
            "scores": [],
            "error": str(e)
        }

# 7. Affichage du classement final
print("\n" + "="*60)
print("CLASSEMENT DES MODÈLES D'EMBEDDING")
print("="*60)

# Trier par score décroissant
sorted_results = sorted(embedding_results.items(), key=lambda x: x[1]["avg_score"], reverse=True)

for rank, (name, data) in enumerate(sorted_results, 1):
    if "error" not in data:
        print(f"{rank}. {name:35} : {data['avg_score']:.2f}/5  ({data['time']:.1f}s)")
    else:
        print(f"{rank}. {name:35} : ERREUR - {data['error']}")

# 8. Sauvegarde des résultats
with open("embedding_comparison.json", "w", encoding="utf-8") as f:
    json.dump(embedding_results, f, indent=2, ensure_ascii=False)

print("\n✅ Résultats sauvegardés dans embedding_comparison.json")

# 9. Sélection du meilleur modèle
if sorted_results:
    best_embedding_name = sorted_results[0][0]
    best_embedding_id = EMBEDDING_MODELS[best_embedding_name]
    best_embedding_score = sorted_results[0][1]["avg_score"]
    
    print("\n" + "="*60)
    print("MEILLEUR MODÈLE D'EMBEDDING")
    print("="*60)
    print(f"🏆 {best_embedding_name}")
    print(f"   ID : {best_embedding_id}")
    print(f"   Score : {best_embedding_score:.2f}/5")
    
    # 10. Optionnel : Créer un retriever avec le meilleur modèle
    print("\n📌 Création du retriever avec le meilleur modèle...")
    best_emb_model = HuggingFaceEmbeddings(
        model_name=best_embedding_id,
        model_kwargs={"device": "cpu"}
    )
    
    splitter_final = RecursiveCharacterTextSplitter(
        chunk_size=test_cs,
        chunk_overlap=test_ov,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks_final = splitter_final.split_documents(all_docs)
    vs_final = Chroma.from_documents(chunks_final, best_emb_model, persist_directory="chroma_best_embedding", collection_name="best")
    retriever_best = vs_final.as_retriever(search_kwargs={"k": test_k})
    
    print("✅ Retriever optimisé avec le meilleur embedding créé.")
else:
    print("\n❌ Aucun modèle n'a pu être testé correctement.")

Configuration de test : chunk_size=400, overlap=16, k=5
Nombre de questions pour le test : 5
Nombre total de pages : 46

TEST DES MODÈLES D'EMBEDDING

📊 Test du modèle : gte-small (léger)
   ID : thenlper/gte-small
  - Chargement du modèle : gte-small (léger)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✅ Score moyen : 3.80/5
   ⏱️  Temps : 741.8 secondes
   📊 Scores individuels : [4, 5, 2, 4, 4]

📊 Test du modèle : gte-large (précis)
   ID : thenlper/gte-large
  - Chargement du modèle : gte-large (précis)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Partie 2 – Industrialisation


In [ ]:
import os
import tempfile
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import ollama

class RAGPipeline:
    def __init__(self, persist_directory="./chroma_app", embedding_model_name="thenlper/gte-small"):
        self.persist_directory = persist_directory
        self.embedding_model = HuggingFaceEmbeddings(
            model_name=embedding_model_name,
            model_kwargs={"device": "cpu"}
        )
        self.vectorstore = None
        self.retriever = None

    def index_documents(self, file_paths):
        """Charge, découpe et indexe une liste de fichiers PDF."""
        all_docs = []
        for file_path in file_paths:
            loader = PyPDFLoader(file_path)
            docs = loader.load()
            all_docs.extend(docs)

        # Utiliser les paramètres optimisés trouvés
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=400,
            chunk_overlap=16,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(all_docs)

        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embedding_model,
            persist_directory=self.persist_directory,
            collection_name="rag_collection"
        )
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})  # meilleur k trouvé

    def delete_all_documents(self):
        """Supprime tous les documents en recréant la collection."""
        if self.vectorstore is not None:
            self.vectorstore.delete_collection()
        self.vectorstore = None
        self.retriever = None

    def query(self, question, model="arcee-ai/trinity-large-preview:free"):
        if self.retriever is None:
            raise ValueError("Aucun document indexé.")
        docs = self.retriever.invoke(question)
        context = "\n\n".join([doc.page_content for doc in docs])
        prompt = f"""Utilise le contexte ci-dessous pour répondre à la question.
Si tu ne connais pas la réponse, dis simplement "Je ne sais pas".

Contexte :
{context}

Question : {question}
Réponse :"""
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
        return {
            "answer": response["message"]["content"],
            "context": context,
            "docs": docs
        }